# 3.4 Dynamic Models (adaptado a lab local)

**Qué se ve aquí:** `@wrap_model_call` intercepta **cada llamada al modelo** y puede cambiar el modelo con `request.override(model=...)`. Aquí: conversación corta → modelo estándar; más de 10 mensajes → modelo con más contexto.

**Local:** estándar = `gemma4:latest` (8K de contexto); grande = `qwen3:8b` con `num_ctx=32768`. `response_metadata["model_name"]` muestra cuál respondió.
> En CPU, cambiar de modelo obliga a Ollama a **cargar el otro** (segundos extra en la primera llamada). En producción el criterio suele ser costo/latencia vs. capacidad.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: Claude Sonnet y "gpt-5-nano" son APIs de pago → dos modelos locales de distinto tamaño y contexto.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
# from langchain.chat_models import init_chat_model
# from typing import Callable
#
# large_model = init_chat_model("claude-sonnet-4-6")
# standard_model = init_chat_model("gpt-5-nano")
#
#
# @wrap_model_call
# def state_based_model(request: ModelRequest, 
# handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
#     """Select model based on State conversation length."""
#     # request.messages is a shortcut for request.state["messages"]
#     message_count = len(request.messages)  
#
#     if message_count > 10:
#         # Long conversation - use model with larger context window
#         model = large_model
#     else:
#         # Short conversation - use efficient model
#         model = standard_model
#
#     request = request.override(model=model)  
#
#     return handler(request)

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: Claude Sonnet y "gpt-5-nano" son APIs de pago → dos modelos locales de distinto tamaño y contexto.
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

large_model = get_model("qwen3:8b", num_ctx=32768)      # "grande": más parámetros y más contexto
standard_model = get_model("gemma4:latest")             # "estándar": el más rápido del lab


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago. (el modelo por defecto; el middleware lo reemplaza en cada llamada)
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     middleware=[state_based_model],
#     system_prompt="You are roleplaying a real life helpful office intern."
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago. (el modelo por defecto; el middleware lo reemplaza en cada llamada)
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
# Sin cambios
print(response["messages"][-1].response_metadata["model_name"])

In [ ]:
# Sin cambios
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
# Sin cambios
print(response["messages"][-1].response_metadata["model_name"])